# Deep Learning of Retinal Ganglion Cell Spatiotemporal Patterns
### Reproducible preprocessing + CNN pipeline

This notebook is the cleaned, ordered version of the full analysis pipeline for the
manuscript *"Deep Learning Reveals Retinal Ganglion Cell Spatiotemporal Patterns
Across Diverse Visual Stimuli."*

**Pipeline (same for every stimulus paradigm):**

```
MEA spike trains (Excel)  ->  per-window spike counts  ->  8x8x3 RGB spatio-temporal maps  ->  task-specific CNN
```

Each of the 64 electrodes maps to one pixel of an 8x8 grid (its physical MEA position).
Three successive response time-windows are written to the Red, Green and Blue channels.
The RGB channels encode **time**, not stimulus wavelength.

**Sections**
1. Setup — imports, configuration, shared helpers
2. Paradigm A — Flash ON/OFF  -> RGB maps
3. Paradigm B — Blue / Green chromatic  -> RGB maps
4. Paradigm C — Chirp (frequency + contrast progression, 10 classes)  -> RGB maps
5. Diagnostics — spike-timeline histogram & window-separability analysis
6. CNN training — Flash ON/OFF (binary)
7. CNN training — Chirp (multi-class frequency and contrast tasks)
8. Per-subject UMAP + HDBSCAN latent-space clustering
9. Handcrafted-feature clustering baseline (ARI / NMI)

All input and output locations are set in the configuration cell of Section 1.


## 1. Setup

Imports, configuration, and shared helper functions used throughout the pipeline.

In [ ]:
import os
import re
import ast
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

### 1.1 Configuration

Set `BASE_DIR` to wherever your data lives. The defaults match a Google Colab run
(`/content`). Each retina is one Excel file in `RAW_XLSX_DIR`, containing two sheets:
`trgss_1` (trigger times: `Flash`, `BG`, `Chirp` columns) and `Original_DataFrame_1`
(spike times, one column per electrode, named like `adch_11a`).

In [ ]:
# ---- EDIT THESE PATHS ----
BASE_DIR     = "/content"                       # root for everything
RAW_XLSX_DIR = os.path.join(BASE_DIR, "new data")   # input: one .xlsx per retina

# Derived output directories (created automatically)
OUT_DIR = os.path.join(BASE_DIR, "outputs")

PATHS = {
    # Flash ON/OFF
    "flash_counts":  os.path.join(OUT_DIR, "flash_counts"),
    "flash_matrix":  os.path.join(OUT_DIR, "flash_matrix"),
    "flash_images":  os.path.join(OUT_DIR, "flash_rgb_images"),
    # Blue / Green chromatic
    "bg_counts":     os.path.join(OUT_DIR, "bg_counts"),
    "bg_matrix":     os.path.join(OUT_DIR, "bg_matrix"),
    "bg_images":     os.path.join(OUT_DIR, "bg_rgb_images"),
    # Chirp (10-class)
    "chirp_counts":  os.path.join(OUT_DIR, "chirp_counts"),
    "chirp_matrix":  os.path.join(OUT_DIR, "chirp_matrix"),
    "chirp_images":  os.path.join(OUT_DIR, "chirp_rgb_images"),
    # Diagnostics
    "diagnostics":   os.path.join(OUT_DIR, "diagnostics"),
}
for p in PATHS.values():
    os.makedirs(p, exist_ok=True)

IMG_OUT_SIZE = (264, 264)   # final PNG size (nearest-neighbour upsample of the 8x8 map)
SEED = 42

### 1.2 Shared helpers

In [ ]:
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass

seed_everything()


def convert_to_float(x):
    '''Extract the first decimal number from a cell, else NaN. Pass through numbers.'''
    if isinstance(x, str):
        match = re.search(r'\d+\.\d+', x)
        return float(match.group()) if match else np.nan
    return x


# Single authoritative electrode -> (row, col) map for the 8x8 MEA grid.
# (Column index = first digit, row index = second digit, suffix 'a' = unit a.)
CHANNEL_MAP = {
    (r, c): f"adch_{c+1}{r+1}a"
    for r in range(8) for c in range(8)
}
# -> {(0,0):'adch_11a', (0,1):'adch_21a', ..., (7,7):'adch_88a'}


def find_xlsx(directory):
    '''Return sorted .xlsx/.xls file paths in a directory.'''
    if not os.path.isdir(directory):
        print(f"WARNING: input directory not found: {directory}")
        return []
    return sorted(
        os.path.join(directory, f)
        for f in os.listdir(directory)
        if f.lower().endswith((".xlsx", ".xls"))
    )


def load_retina(file_path, trigger_col):
    '''Load one retina Excel file. Returns (trigger_times ndarray, spikes DataFrame).'''
    df_trig = pd.read_excel(file_path, sheet_name="trgss_1")
    df_spk  = pd.read_excel(file_path, sheet_name="Original_DataFrame_1")
    triggers = df_trig[trigger_col].apply(convert_to_float).dropna().values
    for col in df_spk.columns:
        df_spk[col] = df_spk[col].apply(convert_to_float)
    return triggers, df_spk


def count(spikes, t0, t1):
    '''Number of spike times in the half-open window [t0, t1).'''
    return int(((spikes >= t0) & (spikes < t1)).sum())


def rgb_counts_to_matrix_csv(in_dir, out_dir, keep=lambda f: f.endswith(".csv")):
    '''Turn each per-channel 'r,g,b' count CSV into an 8x8x3 matrix CSV.'''
    os.makedirs(out_dir, exist_ok=True)
    n = 0
    for file in os.listdir(in_dir):
        if not keep(file):
            continue
        df = pd.read_csv(os.path.join(in_dir, file))
        if df.empty:
            continue
        interval_col = "Intervals" if "Intervals" in df.columns else df.columns[0]
        rows = []
        for idx, row in df.iterrows():
            interval = row[interval_col]
            matrix = np.zeros((8, 8, 3), dtype=float).tolist()
            for (i, j), ch_id in CHANNEL_MAP.items():
                cols = [c for c in df.columns if ch_id in c]
                if cols and not pd.isna(row[cols[0]]):
                    matrix[i][j] = [float(x) for x in str(row[cols[0]]).split(",")]
            rows.append((interval, str(matrix)))
        out = os.path.join(out_dir, file.replace(".csv", "_matrix.csv"))
        pd.DataFrame(rows, columns=["Spike Count Intervals", "Matrix"]).to_csv(out, index=False)
        n += 1
    print(f"  built {n} matrix files -> {out_dir}")


def normalize_matrix(matrix, gmax, gmin):
    '''Min-max scale a matrix to 0..255 using a shared (per-subject) ceiling.'''
    if gmax != gmin:
        norm = (matrix - gmin) / (gmax - gmin) * 255.0
        return np.clip(norm, 0, 255)
    if gmax == 0:
        return np.zeros(matrix.shape)
    return np.full(matrix.shape, 127.0)


def save_images(df, gmax, gmin, out_dir):
    '''Render each stored 8x8(x3) matrix to a PNG, named by its interval.'''
    os.makedirs(out_dir, exist_ok=True)
    saved = 0
    for _, row in df.iterrows():
        matrix = np.array(ast.literal_eval(row["Matrix"]), dtype=np.float32)
        norm = normalize_matrix(matrix, gmax, gmin)
        img = Image.fromarray(np.uint8(norm)).resize(IMG_OUT_SIZE, Image.NEAREST)
        safe = str(row["Spike Count Intervals"]).replace("/", "_").replace("\\", "_")
        img.save(os.path.join(out_dir, f"{safe}.png"))
        saved += 1
    return saved


def load_matrices(df):
    return [np.array(ast.literal_eval(r["Matrix"])) for _, r in df.iterrows()]

## 2. Paradigm A — Flash ON/OFF -> RGB maps

**Stimulus:** repeating 2 s light-ON / 2 s light-OFF cycles.
Each cycle produces two samples: an **ON** map and an **OFF** map.
Within each 2 s phase, three temporal sub-windows become the R, G, B channels
(early transient / trough / sustained for ON; delayed burst / plateau / baseline for OFF).

**Labels for the CNN:** `combined_on` vs `combined_off` (binary).

### 2.1 Spike counts (RGB sub-windows)

In [ ]:
def flash_rgb_counts(df, flash_times):
    cols = [c for c in df.columns if c.startswith("adch_") and c.endswith("a")]
    if not cols:
        return pd.DataFrame(), pd.DataFrame()
    rows_on, rows_off = [], []
    for ft in flash_times:
        on0, off0 = ft, ft + 2.0
        r_on  = {"Intervals": f"On_{on0}-{on0 + 2.0}"}
        r_off = {"Intervals": f"Off_{off0}-{off0 + 2.0}"}
        for ch in cols:
            sp = df[ch].dropna()
            # ON: 0-400ms burst / 400ms-1.0s trough / 1.0-2.0s sustained
            r_on[ch]  = f"{count(sp, on0, on0+0.4)},{count(sp, on0+0.4, on0+1.0)},{count(sp, on0+1.0, on0+2.0)}"
            # OFF: 0-500ms delayed burst / 500ms-1.2s plateau / 1.2-2.0s baseline
            r_off[ch] = f"{count(sp, off0, off0+0.5)},{count(sp, off0+0.5, off0+1.2)},{count(sp, off0+1.2, off0+2.0)}"
        rows_on.append(r_on)
        rows_off.append(r_off)
    return pd.DataFrame(rows_on), pd.DataFrame(rows_off)


for fp in find_xlsx(RAW_XLSX_DIR):
    prefix = os.path.splitext(os.path.basename(fp))[0]
    print(f"Flash counts: {prefix}")
    try:
        flash_times, df_spk = load_retina(fp, "Flash")
        df_on, df_off = flash_rgb_counts(df_spk, flash_times)
        if not df_on.empty:
            df_on.to_csv(os.path.join(PATHS["flash_counts"], f"{prefix}_combined_on.csv"), index=False)
            df_off.to_csv(os.path.join(PATHS["flash_counts"], f"{prefix}_combined_off.csv"), index=False)
    except Exception as e:
        print(f"  error: {e}")

### 2.2 8x8x3 matrices

In [ ]:
rgb_counts_to_matrix_csv(PATHS["flash_counts"], PATHS["flash_matrix"],
                         keep=lambda f: f.endswith(".csv") and "combined" in f)

### 2.3 RGB images (paired ON/OFF normalization per subject)

ON and OFF maps of the same retina are normalized against a **shared** max/min so their
brightness is comparable.

In [ ]:
def build_flash_images():
    files = [f for f in os.listdir(PATHS["flash_matrix"]) if f.endswith("_matrix.csv")]
    prefixes = {f.split("_combined_on")[0].split("_combined_off")[0] for f in files}
    for prefix in sorted(prefixes):
        print(f"Flash images: {prefix}")
        on  = [f for f in files if prefix in f and "combined_on"  in f]
        off = [f for f in files if prefix in f and "combined_off" in f]
        df_on  = pd.read_csv(os.path.join(PATHS["flash_matrix"], on[0]))  if on  else pd.DataFrame()
        df_off = pd.read_csv(os.path.join(PATHS["flash_matrix"], off[0])) if off else pd.DataFrame()

        mats = []
        if not df_on.empty:  mats += load_matrices(df_on)
        if not df_off.empty: mats += load_matrices(df_off)
        if not mats:
            continue
        gmax, gmin = np.max(mats), np.min(mats)

        subj = os.path.join(PATHS["flash_images"], prefix)
        if not df_on.empty:
            save_images(df_on,  gmax, gmin, os.path.join(subj, "combined_on"))
        if not df_off.empty:
            save_images(df_off, gmax, gmin, os.path.join(subj, "combined_off"))
    print("Flash images done.")

build_flash_images()

## 3. Paradigm B — Blue / Green chromatic -> RGB maps

**Stimulus:** alternating 6 s cycles; odd triggers = Green, even triggers = Blue.
Within each cycle, three physiologically chosen windows become R/G/B:
ON response (0.0-1.2 s), sustained (1.5-2.9 s), OFF response (3.0-3.7 s).

**Labels for the CNN:** `green_cycle` vs `blue_cycle`.

### 3.1 Spike counts (RGB windows)

In [ ]:
# Window boundaries (seconds, relative to cycle start)
BG_ON        = (0.00, 1.20)
BG_SUSTAINED = (1.50, 2.90)
BG_OFF       = (3.00, 3.70)


def bg_rgb_counts(df, bg_times):
    cols = [c for c in df.columns if c.startswith("adch_") and c.endswith("a")]
    if not cols:
        return pd.DataFrame(), pd.DataFrame()
    green_starts = bg_times[::2]
    blue_starts  = bg_times[1::2]

    def cycle_rows(starts, label):
        rows = []
        for st in starts:
            row = {"Intervals": f"{label}_{st:.2f}-{st + 6.0:.2f}"}
            for ch in cols:
                sp = df[ch].dropna()
                r = count(sp, st + BG_ON[0],        st + BG_ON[1])
                g = count(sp, st + BG_SUSTAINED[0], st + BG_SUSTAINED[1])
                b = count(sp, st + BG_OFF[0],       st + BG_OFF[1])
                row[ch] = f"{r},{g},{b}"
            rows.append(row)
        return pd.DataFrame(rows)

    return cycle_rows(green_starts, "GreenCycle"), cycle_rows(blue_starts, "BlueCycle")


for fp in find_xlsx(RAW_XLSX_DIR):
    prefix = os.path.splitext(os.path.basename(fp))[0]
    print(f"BG counts: {prefix}")
    try:
        bg_times, df_spk = load_retina(fp, "BG")
        df_green, df_blue = bg_rgb_counts(df_spk, bg_times)
        if not df_green.empty:
            df_green.to_csv(os.path.join(PATHS["bg_counts"], f"{prefix}_green_cycle.csv"), index=False)
        if not df_blue.empty:
            df_blue.to_csv(os.path.join(PATHS["bg_counts"], f"{prefix}_blue_cycle.csv"), index=False)
    except Exception as e:
        print(f"  error: {e}")

### 3.2 8x8x3 matrices

In [ ]:
rgb_counts_to_matrix_csv(PATHS["bg_counts"], PATHS["bg_matrix"])

### 3.3 RGB images (shared Green+Blue normalization per subject)

In [ ]:
def build_bg_images():
    files = [f for f in os.listdir(PATHS["bg_matrix"]) if f.endswith("_matrix.csv")]
    prefixes = {f.split("_green_cycle")[0].split("_blue_cycle")[0] for f in files}
    for prefix in sorted(prefixes):
        print(f"BG images: {prefix}")
        g = [f for f in files if prefix in f and "green_cycle" in f]
        b = [f for f in files if prefix in f and "blue_cycle"  in f]
        df_g = pd.read_csv(os.path.join(PATHS["bg_matrix"], g[0])) if g else pd.DataFrame()
        df_b = pd.read_csv(os.path.join(PATHS["bg_matrix"], b[0])) if b else pd.DataFrame()

        mats = []
        if not df_g.empty: mats += load_matrices(df_g)
        if not df_b.empty: mats += load_matrices(df_b)
        if not mats:
            continue
        gmax, gmin = np.max(mats), np.min(mats)

        subj = os.path.join(PATHS["bg_images"], prefix)
        if not df_g.empty: save_images(df_g, gmax, gmin, os.path.join(subj, "green_cycle"))
        if not df_b.empty: save_images(df_b, gmax, gmin, os.path.join(subj, "blue_cycle"))
    print("BG images done.")

build_bg_images()

## 4. Paradigm C — Chirp -> RGB maps (10 classes)

**Stimulus:** a 32 s chirp. The first 16 s step up in **frequency**
(`flow, fmid, fhigh, fhigher, fhigher1`); the second 16 s step up in **contrast**
(`clow, cmid, chigh, chigher, chigher1`).

Each of the 10 segments is split into three micro-windows -> R/G/B, and counts are
divided by the window width so each channel is a firing *rate*, making the unequal-width
windows comparable.

### 4.1 Spike counts (RGB micro-bins, firing rate)

In [ ]:
CHIRP_CLASSES = ["flow", "fmid", "fhigh", "fhigher", "fhigher1",
                 "clow", "cmid", "chigh", "chigher", "chigher1"]

# (class -> list of three (start_offset, end_offset) micro-windows, seconds from chirp start)
CHIRP_WINDOWS = {
    "flow":     [(0.0, 0.5),   (0.5, 1.2),   (1.2, 2.0)],
    "fmid":     [(2.0, 2.8),   (2.8, 3.8),   (3.8, 5.0)],
    "fhigh":    [(5.0, 5.8),   (5.8, 6.8),   (6.8, 8.0)],
    "fhigher":  [(8.0, 8.8),   (8.8, 9.8),   (9.8, 11.0)],
    "fhigher1": [(11.0, 12.2), (12.2, 13.8), (13.8, 16.0)],
    "clow":     [(16.0, 16.41),(16.41, 16.96),(16.96, 18.0)],
    "cmid":     [(18.0, 18.43),(18.43, 19.59),(19.59, 21.0)],
    "chigh":    [(21.0, 21.27),(21.27, 22.27),(22.27, 24.0)],
    "chigher":  [(24.0, 24.59),(24.59, 25.63),(25.63, 27.0)],
    "chigher1": [(27.0, 27.74),(27.74, 29.07),(29.07, 32.0)],
}


def chirp_rgb_counts(df, start_times):
    cols = [c for c in df.columns if c.startswith("adch_") and c.endswith("a")]
    if not cols:
        return {cls: pd.DataFrame() for cls in CHIRP_CLASSES}

    data = {cls: [] for cls in CHIRP_CLASSES}
    for st in start_times:
        for cls in CHIRP_CLASSES:
            (a0, a1), (b0, b1), (c0, c1) = CHIRP_WINDOWS[cls]
            row = {"Intervals": f"{cls}_{st + a0:.2f}-{st + c1:.2f}"}
            for ch in cols:
                sp = df[ch].dropna()
                r = count(sp, st + a0, st + a1) / (a1 - a0)
                g = count(sp, st + b0, st + b1) / (b1 - b0)
                b = count(sp, st + c0, st + c1) / (c1 - c0)
                row[ch] = f"{r},{g},{b}"
            data[cls].append(row)
    return {cls: pd.DataFrame(rows) for cls, rows in data.items()}


CHIRP_TRIGGER_COLS = ["Chirp", "chirp", "Trigger", "triggers", "ChirpTime"]

for fp in find_xlsx(RAW_XLSX_DIR):
    prefix = os.path.splitext(os.path.basename(fp))[0]
    print(f"Chirp counts: {prefix}")
    try:
        df_trig = pd.read_excel(fp, sheet_name="trgss_1")
        trig_col = next((c for c in CHIRP_TRIGGER_COLS if c in df_trig.columns), None)
        if trig_col is None:
            print("  skipped: no chirp trigger column")
            continue
        start_times = df_trig[trig_col].apply(convert_to_float).dropna().values
        df_spk = pd.read_excel(fp, sheet_name="Original_DataFrame_1")
        for col in df_spk.columns:
            df_spk[col] = df_spk[col].apply(convert_to_float)

        per_class = chirp_rgb_counts(df_spk, start_times)
        for cls, dfc in per_class.items():
            if not dfc.empty:
                dfc.to_csv(os.path.join(PATHS["chirp_counts"], f"{prefix}_{cls}.csv"), index=False)
    except Exception as e:
        print(f"  error: {e}")

### 4.2 8x8x3 matrices

In [ ]:
rgb_counts_to_matrix_csv(PATHS["chirp_counts"], PATHS["chirp_matrix"])

### 4.3 RGB images (one global per-subject ceiling across all 10 classes)

Images are written to `chirp_rgb_images/<subject>/<class>/`, which is exactly the layout
the chirp CNN (Section 7) expects.

In [ ]:
def build_chirp_images():
    files = [f for f in os.listdir(PATHS["chirp_matrix"]) if f.endswith("_matrix.csv")]
    # subject prefix = filename with any _<class> suffix stripped
    prefixes = set()
    for f in files:
        temp = f
        for cls in CHIRP_CLASSES:
            temp = temp.split(f"_{cls}")[0]
        prefixes.add(temp)

    for prefix in sorted(prefixes):
        print(f"Chirp images: {prefix}")
        subj_dfs, mats = {}, []
        for cls in CHIRP_CLASSES:
            match = [f for f in files if prefix in f and f"_{cls}_" in f]
            if match:
                dfc = pd.read_csv(os.path.join(PATHS["chirp_matrix"], match[0]))
                subj_dfs[cls] = dfc
                mats += load_matrices(dfc)
            else:
                subj_dfs[cls] = pd.DataFrame()
        if not mats:
            continue
        gmax, gmin = np.max(mats), np.min(mats)
        for cls, dfc in subj_dfs.items():
            if not dfc.empty:
                save_images(dfc, gmax, gmin, os.path.join(PATHS["chirp_images"], prefix, cls))
    print("Chirp images done.")

build_chirp_images()

## 5. Diagnostics (supporting figures)

These do not feed the CNN; they justify the temporal window choices used above.

### 5.1 Population spike timeline across the 4 s flash cycle

In [ ]:
def plot_flash_timeline(max_subjects=5):
    all_rel = []
    for fp in find_xlsx(RAW_XLSX_DIR)[:max_subjects]:
        try:
            flash_times, df_spk = load_retina(fp, "Flash")
            cols = [c for c in df_spk.columns if c.startswith("adch_") and c.endswith("a")]
            for ch in cols:
                sp = df_spk[ch].dropna().values
                for ft in flash_times:
                    w = sp[(sp >= ft) & (sp < ft + 4.0)]
                    all_rel.extend(w - ft)
        except Exception as e:
            print(f"  skip {os.path.basename(fp)}: {e}")
    if not all_rel:
        print("No spikes extracted.")
        return
    plt.figure(figsize=(14, 6))
    plt.hist(all_rel, bins=40, range=(0, 4), color="purple", edgecolor="black", alpha=0.7)
    plt.axvspan(0, 2, color="yellow", alpha=0.15, label="Light ON (2 s)")
    plt.axvspan(2, 4, color="gray",   alpha=0.15, label="Light OFF (2 s)")
    plt.axvline(2.0, color="red", lw=2, label="Light switches OFF")
    plt.title("Population spike distribution across the 4 s flash cycle", fontweight="bold")
    plt.xlabel("Time since flash trigger (s)"); plt.ylabel("Total spikes")
    plt.xlim(0, 4); plt.grid(axis="y", ls=":", alpha=0.6); plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(PATHS["diagnostics"], "flash_timeline.png"), dpi=300, bbox_inches="tight")
    plt.show()

plot_flash_timeline()

### 5.2 Blue/Green window-separability index + PCA

For each 100 ms window we compute how separable Green vs Blue cycles are (|AUC-0.5|),
then project the full temporal profiles with PCA.

In [ ]:
def bg_separability(max_subjects=5, bin_size=0.1, cycle=6.0):
    from sklearn.metrics import roc_auc_score
    from sklearn.preprocessing import StandardScaler
    from sklearn.decomposition import PCA

    bins = np.arange(0, cycle + bin_size, bin_size)
    n_bins = len(bins) - 1
    X, y = [], []
    for fp in find_xlsx(RAW_XLSX_DIR)[:max_subjects]:
        try:
            bg_times, df_spk = load_retina(fp, "BG")
            cols = [c for c in df_spk.columns if c.startswith("adch_") and c.endswith("a")]
            spikes = df_spk[cols].values.flatten()
            spikes = np.sort(spikes[~np.isnan(spikes)])
            for idx, st in enumerate(bg_times):
                lo = np.searchsorted(spikes, st)
                hi = np.searchsorted(spikes, st + cycle)
                prof, _ = np.histogram(spikes[lo:hi] - st, bins=bins)
                X.append(prof)
                y.append(0 if idx % 2 == 0 else 1)  # 0 Green, 1 Blue
        except Exception as e:
            print(f"  skip {os.path.basename(fp)}: {e}")
    X, y = np.array(X), np.array(y)
    if X.size == 0 or len(np.unique(y)) < 2:
        print("Not enough data / only one class.")
        return

    auc = []
    for b in range(n_bins):
        try:
            a = roc_auc_score(y, X[:, b]); auc.append(max(a, 1 - a))
        except Exception:
            auc.append(0.5)
    auc = np.array(auc)
    t = bins[:-1] + bin_size / 2

    X_pca = PCA(n_components=2).fit(StandardScaler().fit_transform(X))
    coords = X_pca.transform(StandardScaler().fit_transform(X))

    fig, ax = plt.subplots(2, 1, figsize=(12, 10))
    ax[0].plot(t, auc * 100, color="#E94F37", lw=2.5)
    ax[0].axvline(3.0, color="black", ls="--", alpha=0.7, label="Light OFF")
    ax[0].set_title("Green vs Blue separability (100 ms windows)", fontweight="bold")
    ax[0].set_xlabel("Time since trigger (s)"); ax[0].set_ylabel("Separability (%)")
    ax[0].set_ylim(45, 105); ax[0].legend()
    ax[1].scatter(coords[y == 0, 0], coords[y == 0, 1], c="#2EC4B6", alpha=0.6, edgecolors="w", label="Green")
    ax[1].scatter(coords[y == 1, 0], coords[y == 1, 1], c="#FF9F1C", alpha=0.6, edgecolors="w", label="Blue")
    ax[1].set_title("PCA of 6 s temporal profiles", fontweight="bold")
    ax[1].set_xlabel(f"PC1 ({X_pca.explained_variance_ratio_[0]*100:.1f}%)")
    ax[1].set_ylabel(f"PC2 ({X_pca.explained_variance_ratio_[1]*100:.1f}%)")
    ax[1].legend()
    plt.tight_layout()
    plt.savefig(os.path.join(PATHS["diagnostics"], "bg_separability.png"), dpi=300, bbox_inches="tight")
    plt.show()

bg_separability()

## 6. CNN training — Flash ON/OFF (binary)

Subject-independent evaluation: 5-fold **GroupKFold** (no retina appears in both train
and validation) plus **LOSO** (leave-one-subject-out). Learning curves and confusion
matrices are saved per fold.

`DATA_PATH` must point to the folder of subject directories produced in Section 2.3,
each containing `combined_on/` and `combined_off/`.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import GroupKFold

# ---- Flash training config ----
FLASH_DATA_PATH = PATHS["flash_images"]   # subjects/<id>/{combined_on,combined_off}/*.png
FLASH_OUT       = os.path.join(OUT_DIR, "flash_cnn")
os.makedirs(FLASH_OUT, exist_ok=True)

IMG_SIZE = 128
BATCH_SIZE = 32
NUM_EPOCHS = 100
LR = 1e-4
WEIGHT_DECAY = 1e-3
EARLY_STOP_PATIENCE = 30
N_FOLDS = 5
TARGET_CLASSES = ["combined_off", "combined_on"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything()

In [ ]:
def load_flash_dataframe(data_path, classes):
    rows = []
    for subject in os.listdir(data_path):
        sp = os.path.join(data_path, subject)
        if not os.path.isdir(sp):
            continue
        for phase in classes:
            pp = os.path.join(sp, phase)
            if os.path.isdir(pp):
                for f in os.listdir(pp):
                    if f.lower().endswith((".png", ".jpg", ".jpeg")):
                        rows.append({"subject_id": subject, "label": phase,
                                     "image_path": os.path.join(pp, f)})
    return pd.DataFrame(rows)


class SpatialDataset(Dataset):
    def __init__(self, dataframe, transform, classes):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.label_map = {classes[0]: 0, classes[1]: 1}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["image_path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(self.label_map[row["label"]], dtype=torch.long)


train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(8),
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.95, 1.05)),
    transforms.ToTensor(),
])
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])


class BasicCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(0.6),
            nn.Linear(32, 2),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [ ]:
def run_flash_fold(train_df, val_df, fold_id, prefix, extra=""):
    print(f"\n{'='*60}\nTraining {fold_id}{extra}")
    print(f"Train subjects: {train_df['subject_id'].nunique()} | "
          f"Val subjects: {val_df['subject_id'].nunique()}\n{'='*60}")

    train_loader = DataLoader(SpatialDataset(train_df, train_transform, TARGET_CLASSES),
                              batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                              num_workers=2, pin_memory=True)
    val_loader = DataLoader(SpatialDataset(val_df, val_transform, TARGET_CLASSES),
                            batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    model = BasicCNN().to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.12)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=4)

    best_acc, best_epoch, patience = 0.0, 0, 0
    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    ckpt = os.path.join(FLASH_OUT, f"best_model_{prefix}.pth")

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        run_loss = correct = total = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            run_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)
        tr_loss, tr_acc = run_loss / total, 100 * correct / total

        model.eval()
        v_loss = correct = total = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                out = model(x)
                v_loss += criterion(out, y).item() * x.size(0)
                correct += (out.argmax(1) == y).sum().item()
                total += y.size(0)
        va_loss, va_acc = v_loss / total, 100 * correct / total

        for k, v in zip(hist, [tr_loss, va_loss, tr_acc, va_acc]):
            hist[k].append(v)
        scheduler.step(va_acc)

        if va_acc > best_acc:
            best_acc, best_epoch, patience = va_acc, epoch, 0
            torch.save(model.state_dict(), ckpt)
        else:
            patience += 1
        print(f"{fold_id} | Ep {epoch:3d} | Tr {tr_acc:6.2f}% | Val {va_acc:6.2f}% | "
              f"LR {optimizer.param_groups[0]['lr']:.6f}")
        if patience >= EARLY_STOP_PATIENCE:
            print(f"  early stop @ {epoch}")
            break
    print(f">>> {fold_id} best val acc: {best_acc:.2f}% (epoch {best_epoch})")

    # --- plots from best checkpoint ---
    model.load_state_dict(torch.load(ckpt, map_location=device))
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for x, y in val_loader:
            preds.extend(model(x.to(device)).argmax(1).cpu().numpy())
            labels.extend(y.numpy())

    sns.set_style("whitegrid")
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    ax[0].plot(hist["train_loss"], label="Train", lw=2); ax[0].plot(hist["val_loss"], label="Val", lw=2)
    ax[0].set_title(f"{fold_id} Loss{extra}"); ax[0].legend()
    ax[1].plot(hist["train_acc"], label="Train", lw=2); ax[1].plot(hist["val_acc"], label="Val", lw=2)
    ax[1].set_title(f"{fold_id} Accuracy{extra}"); ax[1].legend()
    plt.tight_layout(); plt.savefig(os.path.join(FLASH_OUT, f"{prefix}_curves.png"), dpi=300, bbox_inches="tight"); plt.close()

    cm = confusion_matrix(labels, preds)
    ConfusionMatrixDisplay(cm, display_labels=TARGET_CLASSES).plot(cmap="Blues", values_format="d")
    plt.title(f"{fold_id} Confusion Matrix{extra}")
    plt.savefig(os.path.join(FLASH_OUT, f"{prefix}_cm.png"), dpi=300, bbox_inches="tight"); plt.close()
    return best_acc

In [ ]:
# ---- Run Flash training (5-fold GroupKFold, then LOSO) ----
df_flash = load_flash_dataframe(FLASH_DATA_PATH, TARGET_CLASSES)
print(f"Loaded {len(df_flash)} images from {df_flash['subject_id'].nunique()} subjects.")

# 5-fold GroupKFold
gkf = GroupKFold(n_splits=N_FOLDS)
fold_acc = []
for k, (tr, va) in enumerate(gkf.split(df_flash, groups=df_flash["subject_id"])):
    fold_acc.append(run_flash_fold(df_flash.iloc[tr], df_flash.iloc[va],
                                   f"Fold_{k+1}", f"fold{k+1}"))
print(f"\n5-FOLD: {np.mean(fold_acc):.2f}% +/- {np.std(fold_acc, ddof=1):.2f}%  {[round(a,2) for a in fold_acc]}")

# LOSO (leave-one-subject-out)
n_subj = df_flash["subject_id"].nunique()
loso = GroupKFold(n_splits=n_subj)
loso_acc = []
for k, (tr, va) in enumerate(loso.split(df_flash, groups=df_flash["subject_id"])):
    left_out = df_flash.iloc[va]["subject_id"].iloc[0]
    loso_acc.append(run_flash_fold(df_flash.iloc[tr], df_flash.iloc[va],
                                   f"LOSO_{k+1}", f"loso{k+1}", f" (left out: {left_out})"))
print(f"\nLOSO: {np.mean(loso_acc):.2f}% +/- {np.std(loso_acc, ddof=1):.2f}%")

## 7. CNN training — Chirp (multi-class tasks)

A compact 3-conv-block network trained with **StratifiedGroupKFold** (subject-independent,
class-balanced). Each task is a different subset of the 10 chirp classes.
`CHIRP_DATA_ROOT` must point to the images from Section 4.3
(`<subject>/<class>/*.png`).

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

CHIRP_DATA_ROOT = PATHS["chirp_images"]
CHIRP_OUT = os.path.join(OUT_DIR, "chirp_cnn")
os.makedirs(CHIRP_OUT, exist_ok=True)

C_IMG_SIZE = 128
C_BATCH = 32
C_EPOCHS = 50
C_LR = 3e-4
C_WD = 0.02
C_FOLDS = 5
seed_everything()


class ChirpCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


class ChirpDataset(Dataset):
    def __init__(self, dataframe, transform, label_map):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.label_map = label_map

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["image_path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(self.label_map[row["label"]], dtype=torch.long)


chirp_train_tf = transforms.Compose([
    transforms.Resize((C_IMG_SIZE, C_IMG_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.5),
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08)),
    transforms.ToTensor(),
])
chirp_val_tf = transforms.Compose([
    transforms.Resize((C_IMG_SIZE, C_IMG_SIZE)),
    transforms.ToTensor(),
])

In [ ]:
def load_chirp_dataframe(root):
    '''Walk <root>/<subject>/<class>/*.png. Subject = the folder under root.'''
    rows = []
    for root_dir, _, files in os.walk(root):
        for f in files:
            if not f.lower().endswith((".png", ".jpg", ".jpeg")):
                continue
            full = os.path.join(root_dir, f)
            cls = os.path.basename(os.path.dirname(full))
            rel = os.path.relpath(full, root)
            subject = rel.split(os.sep)[0]       # first folder below root
            rows.append({"subject_id": subject, "label": cls, "image_path": full})
    return pd.DataFrame(rows, columns=["subject_id", "label", "image_path"])


def run_chirp_task(task_name, class_list, full_df):
    print(f"\n{'='*70}\nTASK: {task_name} | classes: {class_list}\n{'='*70}")
    task_df = full_df[full_df["label"].isin(class_list)].copy()
    if task_df.empty:
        print("  no images for this task."); return
    print(task_df["label"].value_counts().to_string())

    label_map = {c: i for i, c in enumerate(class_list)}
    save_dir = os.path.join(CHIRP_OUT, "_".join(class_list))
    os.makedirs(save_dir, exist_ok=True)

    gkf = StratifiedGroupKFold(n_splits=C_FOLDS, shuffle=True, random_state=SEED)
    accs = []
    for fold, (tr, va) in enumerate(gkf.split(task_df, task_df["label"], groups=task_df["subject_id"])):
        train_df, val_df = task_df.iloc[tr], task_df.iloc[va]
        train_loader = DataLoader(ChirpDataset(train_df, chirp_train_tf, label_map),
                                  batch_size=C_BATCH, shuffle=True, drop_last=True)
        val_loader = DataLoader(ChirpDataset(val_df, chirp_val_tf, label_map),
                                batch_size=C_BATCH, shuffle=False)

        model = ChirpCNN(num_classes=len(class_list)).to(device)
        counts_ = train_df["label"].value_counts().reindex(class_list).fillna(0).values
        weights = torch.tensor(1.0 / (counts_ + 1e-6), dtype=torch.float32).to(device)
        criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.15)
        optimizer = optim.AdamW(model.parameters(), lr=C_LR, weight_decay=C_WD)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=C_EPOCHS)

        best_acc = 0.0
        hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
        ckpt = os.path.join(save_dir, f"best_{task_name}_fold{fold+1}.pth")

        for epoch in range(1, C_EPOCHS + 1):
            model.train()
            rl = c = t = 0
            for x, y in train_loader:
                x, y = x.to(device), y.to(device)
                optimizer.zero_grad()
                out = model(x); loss = criterion(out, y)
                loss.backward(); optimizer.step()
                rl += loss.item() * x.size(0); c += (out.argmax(1) == y).sum().item(); t += y.size(0)
            tr_loss, tr_acc = rl / t, 100 * c / t

            model.eval(); vl = c = t = 0
            with torch.no_grad():
                for x, y in val_loader:
                    x, y = x.to(device), y.to(device)
                    out = model(x)
                    vl += criterion(out, y).item() * x.size(0)
                    c += (out.argmax(1) == y).sum().item(); t += y.size(0)
            va_loss, va_acc = vl / t, 100 * c / t
            for k, v in zip(hist, [tr_loss, va_loss, tr_acc, va_acc]):
                hist[k].append(v)
            scheduler.step()
            if va_acc > best_acc:
                best_acc = va_acc
                torch.save(model.state_dict(), ckpt)
            print(f"  Fold {fold+1} Ep {epoch:3d}/{C_EPOCHS} | Tr {tr_acc:6.2f}% | Val {va_acc:6.2f}%")
        print(f">>>> Fold {fold+1} best: {best_acc:.2f}%")
        accs.append(best_acc)

        # confusion matrix + curves from best checkpoint
        model.load_state_dict(torch.load(ckpt, map_location=device)); model.eval()
        preds, labels = [], []
        with torch.no_grad():
            for x, y in val_loader:
                preds.extend(model(x.to(device)).argmax(1).cpu().numpy()); labels.extend(y.numpy())
        plt.figure(figsize=(12, 5))
        plt.subplot(1, 2, 1); plt.plot(hist["train_loss"], label="Train"); plt.plot(hist["val_loss"], label="Val")
        plt.title(f"{task_name} Fold {fold+1} Loss"); plt.legend()
        plt.subplot(1, 2, 2); plt.plot(hist["train_acc"], label="Train"); plt.plot(hist["val_acc"], label="Val")
        plt.title(f"{task_name} Fold {fold+1} Acc"); plt.legend(); plt.tight_layout()
        plt.savefig(os.path.join(save_dir, f"{task_name}_fold{fold+1}_curves.png"), dpi=300, bbox_inches="tight"); plt.close()
        ConfusionMatrixDisplay(confusion_matrix(labels, preds), display_labels=class_list).plot(cmap="Blues")
        plt.title(f"{task_name} Fold {fold+1} CM")
        plt.savefig(os.path.join(save_dir, f"{task_name}_fold{fold+1}_cm.png"), dpi=300, bbox_inches="tight"); plt.close()

    print(f">>> {task_name}: {np.mean(accs):.2f}% +/- {np.std(accs, ddof=1):.2f}%")

In [ ]:
# ---- Run chirp tasks ----
chirp_df = load_chirp_dataframe(CHIRP_DATA_ROOT)
print(f"Loaded {len(chirp_df)} chirp images | classes: {sorted(chirp_df['label'].unique())}")

CHIRP_TASKS = [
    ("freq_progression_5class", ["flow", "fmid", "fhigh", "fhigher", "fhigher1"]),
    ("freq_progression_4class_a", ["flow", "fmid", "fhigher", "fhigher1"]),
    ("freq_progression_4class_b", ["flow", "fmid", "fhigh", "fhigher1"]),
    ("freq_progression_4class_c", ["fmid", "fhigh", "fhigher", "fhigher1"]),
    ("freq_progression_4class_d", ["flow", "fhigh", "fhigher", "fhigher1"]),
    ("freq_low_mid_high",   ["flow", "fhigh", "fmid"]),
    ("freq_low_high_higher",["flow", "fhigh", "fhigher"]),
    ("freq_mid_high_higher",["fmid", "fhigh", "fhigher"]),
    ("contrast_progression_max1", ["clow", "cmid", "chigher1"]),
    ("contrast_progression_max0", ["clow", "cmid", "chigher"]),
    ("contrast_high_end",   ["chigh", "chigher", "chigher1"]),
    ("mixed_extreme",       ["flow", "fhigher1", "chigher1"]),
    ("mixed_mid_range",     ["fmid", "cmid", "chigh"]),
]

for name, classes in CHIRP_TASKS:
    run_chirp_task(name, classes, chirp_df)

## 8. Per-subject UMAP + HDBSCAN latent-space clustering

For each retina, the trained CNN is used as a feature extractor: its convolutional features
(128-dimensional) are embedded with UMAP and clustered with HDBSCAN. Per subject we report the
number of clusters, noise ratio, mean cluster purity, silhouette score, and whether a strong
(>=90% purity) cluster exists for each class. A presence bar plot and example UMAP embeddings
are also produced (Supplementary Tables S2-S4 and Figs S3-S5).

Set the paradigm and the trained-model checkpoint in the configuration block.

In [ ]:
import warnings
import seaborn as sns
from sklearn.metrics import silhouette_score

# ---- CONFIG: paradigm, class folders, model, and trained checkpoint ----
CLUST_TASK       = "flash_on_off"
CLUST_DATA_ROOT  = PATHS["flash_images"]                  # <subject>/<class>/*.png
CLUST_CLASSES    = ["combined_off", "combined_on"]
CLUST_MODEL      = lambda: BasicCNN()                     # RGB model from Section 6
CLUST_MODEL_PATH = os.path.join(FLASH_OUT, "best_model_fold1.pth")
CLUST_IMG_SIZE   = IMG_SIZE
STRONG_THRESHOLD = 0.90
CLUST_SAVE = os.path.join(OUT_DIR, "clustering", CLUST_TASK)
os.makedirs(CLUST_SAVE, exist_ok=True)

# For the chirp paradigm instead, use the multi-class ChirpCNN from Section 7, e.g.:
# CLUST_TASK="chirp_freq"; CLUST_DATA_ROOT=PATHS["chirp_images"]
# CLUST_CLASSES=["flow","fmid","fhigh","fhigher","fhigher1"]
# CLUST_MODEL=lambda: ChirpCNN(num_classes=len(CLUST_CLASSES))
# CLUST_MODEL_PATH=os.path.join(CHIRP_OUT, "_".join(CLUST_CLASSES), f"best_chirp_freq_fold1.pth")


class ClustDataset(Dataset):
    def __init__(self, dataframe, transform, label_map):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.label_map = label_map

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["image_path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(self.label_map[row["label"]], dtype=torch.long)


def load_cluster_data(data_root, classes):
    '''Walk <data_root>/<subject>/<class>/*.png; label = class folder, subject = top folder.'''
    rows = []
    for root, _, files in os.walk(data_root):
        for f in files:
            if not f.lower().endswith((".png", ".jpg", ".jpeg")):
                continue
            full = os.path.join(root, f)
            cls = os.path.basename(os.path.dirname(full))
            if cls not in classes:
                continue
            subject = os.path.relpath(full, data_root).split(os.sep)[0]
            rows.append({"subject_id": subject, "label": cls, "image_path": full})
    return pd.DataFrame(rows)


def run_spatial_clustering(data_root, classes, model_builder, model_path, task, strong_threshold=0.90):
    try:
        import umap, hdbscan
    except ImportError:
        print("Install first:  pip install umap-learn hdbscan"); return

    df = load_cluster_data(data_root, classes)
    if df.empty:
        print(f"No images under {data_root}"); return
    if not os.path.exists(model_path):
        print(f"Trained model not found: {model_path}"); return
    print(f"{task}: {len(df)} images\n{df['label'].value_counts().to_string()}")

    label_map = {c: i for i, c in enumerate(classes)}
    tf = transforms.Compose([transforms.Resize((CLUST_IMG_SIZE, CLUST_IMG_SIZE)), transforms.ToTensor()])
    loader = DataLoader(ClustDataset(df, tf, label_map), batch_size=64, shuffle=False)

    model = model_builder().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    model.eval()

    feats, labs, subs = [], [], []
    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            f = model.features(x.to(device))
            f = f.view(f.size(0), -1)
            feats.append(f.cpu().numpy()); labs.append(y.numpy())
            s = i * loader.batch_size
            subs.append(df["subject_id"].iloc[s:s + x.size(0)].values)
    features = np.concatenate(feats); labels = np.concatenate(labs); subjects = np.concatenate(subs)

    uniq = sorted(np.unique(subjects))
    anon = {o: f"Subject {i+1}" for i, o in enumerate(uniq)}
    anon_to_orig = {v: k for k, v in anon.items()}

    presence, table = [], []
    for subj in uniq:
        m = subjects == subj
        fsub, lsub, n = features[m], labels[m], int(m.sum())
        strong = {c: 0 for c in classes}
        purities, n_clusters, noise_ratio, sil = [], 0, 0.0, np.nan
        if n >= 15:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                emb = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=SEED).fit_transform(fsub)
            cl = hdbscan.HDBSCAN(min_cluster_size=5).fit_predict(emb)
            noise_ratio = (cl == -1).sum() / n
            n_clusters = len(set(cl)) - (1 if -1 in cl else 0)
            for c in set(cl):
                if c == -1:
                    continue
                cm = cl == c
                counts = np.bincount(lsub[cm], minlength=len(classes))
                dom = int(np.argmax(counts)); p = counts[dom] / cm.sum()
                purities.append(p)
                if p >= strong_threshold:
                    strong[classes[dom]] = 1
            nn_mask = cl != -1
            if nn_mask.sum() > 1 and len(set(cl[nn_mask])) > 1:
                try:
                    sil = silhouette_score(emb[nn_mask], cl[nn_mask])
                except Exception:
                    sil = np.nan
        avg_purity = float(np.mean(purities)) if purities else 0.0
        presence.append({"Subject": anon[subj], **{f"strong_{c}": strong[c] for c in classes}})
        row = {"Subject ID": anon[subj], "Samples (n)": n, "Clusters": n_clusters,
               "Noise Ratio": round(noise_ratio, 4), "Avg. Purity": round(avg_purity, 3),
               "Silhouette Score": round(sil, 3) if not np.isnan(sil) else "—"}
        for c in classes:
            row[f"Strong {c} (>=90%)"] = "Yes" if strong[c] else "No"
        if len(classes) == 2:
            row["Has Both"] = "Yes" if all(strong[c] for c in classes) else "No"
        table.append(row)

    table_df = pd.DataFrame(table)
    table_df["num"] = table_df["Subject ID"].str.extract(r"(\d+)").astype(int)
    table_df = table_df.sort_values("num").drop(columns="num")
    print("\n" + "=" * 90)
    print(f"Per-retina spatial cluster stability & purity - {task}")
    print(table_df.to_string(index=False))
    table_df.to_csv(os.path.join(CLUST_SAVE, f"{task}_cluster_summary.csv"), index=False)

    # presence bar plot
    pres = pd.DataFrame(presence)
    plot_df = pres.melt(id_vars="Subject", var_name="Cluster Type", value_name="Exists")
    plot_df["Cluster Type"] = plot_df["Cluster Type"].str.replace("strong_", "")
    plot_df["num"] = plot_df["Subject"].str.extract(r"(\d+)").astype(int)
    plot_df = plot_df.sort_values("num")
    plt.figure(figsize=(14, 5.5))
    sns.barplot(data=plot_df, x="Subject", y="Exists", hue="Cluster Type",
                edgecolor="black", linewidth=0.6)
    plt.title(f"Presence of strong clusters per retina - {task}")
    plt.ylim(0, 1.15); plt.xticks(rotation=45, ha="right"); plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(CLUST_SAVE, f"{task}_cluster_presence_bars.png"), dpi=300, bbox_inches="tight")
    plt.close()

    # example UMAPs: best vs weaker retinas by purity
    sel = table_df.sort_values("Avg. Purity", ascending=False)
    chosen = sel["Subject ID"].head(3).tolist() + sel["Subject ID"].tail(2).tolist()
    fig, axes = plt.subplots(2, 3, figsize=(15, 8.5)); axes = axes.flatten()
    for i, a in enumerate(chosen):
        m = subjects == anon_to_orig[a]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            emb = umap.UMAP(n_neighbors=12, min_dist=0.1, random_state=SEED).fit_transform(features[m])
        sns.scatterplot(x=emb[:, 0], y=emb[:, 1], hue=labels[m], ax=axes[i], s=35, alpha=0.8, legend=(i == 0))
        pur = sel.loc[sel["Subject ID"] == a, "Avg. Purity"].values[0]
        axes[i].set_title(f"{a} (purity {pur:.2f})"); axes[i].set_xlabel(""); axes[i].set_ylabel("")
    axes[5].axis("off")
    plt.suptitle(f"Example UMAPs: good vs weaker retinas - {task}", y=0.98)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(os.path.join(CLUST_SAVE, f"{task}_example_umaps.png"), dpi=300, bbox_inches="tight")
    plt.close()

    pd.DataFrame(list(anon.items()), columns=["original_id", "anonymous_id"]).to_csv(
        os.path.join(CLUST_SAVE, f"{task}_subject_mapping.csv"), index=False)
    print(f"Saved results to {CLUST_SAVE}")
    return table_df


run_spatial_clustering(CLUST_DATA_ROOT, CLUST_CLASSES, CLUST_MODEL, CLUST_MODEL_PATH, CLUST_TASK)

## 9. Handcrafted-feature clustering baseline — Supplementary Table S1

Handcrafted global statistical features are extracted from each spatial activity map,
reduced with PCA, and clustered with K-Means, Agglomerative Ward, and Gaussian Mixture
models. Clustering quality is scored against the stimulus labels with the Adjusted Rand
Index (ARI) and Normalized Mutual Information (NMI). Set `DATASET_TYPE` to `"flash"`,
`"bg"`, or `"chirp"` and run once per paradigm; results and figures are written to
`Clustering_Performance.csv` and the `paper_figures` folder.

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (adjusted_rand_score, normalized_mutual_info_score,
                             confusion_matrix)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from scipy.stats import skew, kurtosis
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ==================== CONFIG ====================
DATASET_TYPE = "flash"          # "flash" | "bg" | "chirp"  (run once per paradigm)

# Input image folders and results directory
_BASE = {"flash": PATHS["flash_images"], "bg": PATHS["bg_images"], "chirp": PATHS["chirp_images"]}
base_path = _BASE[DATASET_TYPE]
results_dir = os.path.join(OUT_DIR, f"cluster_analysis_{DATASET_TYPE}")
os.makedirs(results_dir, exist_ok=True)

paper_dir = os.path.join(results_dir, "paper_figures")
os.makedirs(paper_dir, exist_ok=True)

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300

print(f"\n{'='*80}")
print(f"PAPER-READY ANALYSIS - {DATASET_TYPE.upper()} + OUTLIER DETECTION")
print(f"{'='*80}")

# ==================== DATA LOADING ====================
print("\n[1/7] Loading data...")

data_rows = []

if DATASET_TYPE in ["bg", "flash"]:
    for sample in os.listdir(base_path):
        sample_path = os.path.join(base_path, sample)
        if not os.path.isdir(sample_path): continue
        for condition in os.listdir(sample_path):
            cond_path = os.path.join(sample_path, condition)
            if not os.path.isdir(cond_path): continue
            for img_name in os.listdir(cond_path):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.tif')):
                    data_rows.append({
                        'label': condition.lower(),
                        'image_path': os.path.join(cond_path, img_name),
                        'sample': sample
                    })
else:
    subjects = [d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d))]
    for retina in subjects:
        for cls in os.listdir(os.path.join(base_path, retina)):
            cls_path = os.path.join(base_path, retina, cls)
            if os.path.isdir(cls_path):
                for img_name in os.listdir(cls_path):
                    data_rows.append({'label': cls, 'image_path': os.path.join(cls_path, img_name)})

df = pd.DataFrame(data_rows)
print(f"Total images: {len(df)}")
print(f"Classes: {sorted(df['label'].unique())}")

# ==================== FEATURE EXTRACTION ====================
def extract_rich_features(path):
    img = Image.open(path).convert('L')
    arr = np.array(img, dtype=np.float32) / 255.0
    flat = arr.flatten()
    mean_val = np.mean(arr)
    std_val = np.std(arr) + 1e-8

    feats = []
    feats.append(mean_val)
    feats.append(std_val)
    feats.append(np.min(arr))
    feats.append(np.max(arr))
    feats.append(np.median(arr))

    for p in [5, 10, 25, 75, 90, 95, 99]:
        feats.append(np.percentile(arr, p))

    if std_val > 1e-6:
        feats.append(skew(flat))
        feats.append(kurtosis(flat))
    else:
        feats.append(0.0)
        feats.append(0.0)

    row_means, row_stds = np.mean(arr, axis=1), np.std(arr, axis=1)
    col_means, col_stds = np.mean(arr, axis=0), np.std(arr, axis=0)
    feats.extend([np.mean(row_means), np.std(row_means), np.mean(row_stds), np.std(row_stds)])
    feats.extend([np.mean(col_means), np.std(col_means), np.mean(col_stds), np.std(col_stds)])

    peaks, _ = find_peaks(flat, height=mean_val + 0.6*std_val, distance=8)
    feats.append(len(peaks))
    feats.append(np.mean(flat[peaks]) if len(peaks) > 0 else mean_val)

    hist64, _ = np.histogram(arr, bins=64, range=(0, 1), density=True)
    feats.extend(hist64.tolist())

    h256, _ = np.histogram(arr, bins=256, range=(0, 1), density=True)
    h256 = h256[h256 > 0]
    entropy = -np.sum(h256 * np.log2(h256 + 1e-12)) if len(h256) > 0 else 0.0
    feats.append(entropy)
    feats.append(np.mean(arr == 0))
    feats.append(np.mean(arr > 0.05))
    feats.append(np.sum(arr**2) / arr.size)

    feats.append(np.max(arr) - np.min(arr))
    feats.append(np.percentile(arr, 75) - np.percentile(arr, 25))

    return np.array(feats)

print("\n[2/7] Extracting features...")
X_list, y_list = [], []
for _, row in df.iterrows():
    X_list.append(extract_rich_features(row['image_path']))
    y_list.append(row['label'])

X = np.array(X_list)
y = np.array(y_list)
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

le = LabelEncoder()
y_encoded = le.fit_transform(y)
label_names = list(le.classes_)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

class_features = {cls: X_scaled[y == cls].mean(axis=0) for cls in label_names}

# ==================== OUTLIER DETECTION ====================
print("\n[3/7] Detecting outliers...")
iso = IsolationForest(contamination=0.05, random_state=42, n_estimators=200)
outlier_pred = iso.fit_predict(X_scaled)
df['is_outlier'] = (outlier_pred == -1).astype(int)

print(f"Total outliers: {df['is_outlier'].sum()} ({df['is_outlier'].sum()/len(df)*100:.1f}%)")
print("Outliers per class:")
print(df.groupby('label')['is_outlier'].sum())

df.to_csv(f"{results_dir}/data_with_outlier_flags.csv", index=False)

# ==================== SIMILARITY + CLUSTERING ====================
print("\n[4/7] Computing similarity & clustering...")

sim_matrix = pd.DataFrame(index=label_names, columns=label_names, dtype=float)
for c1 in label_names:
    for c2 in label_names:
        if c1 == c2:
            sim_matrix.loc[c1, c2] = 100.0
        else:
            v1, v2 = class_features[c1], class_features[c2]
            cos = cosine_similarity([v1], [v2])[0][0] * 100
            euc_sim = 100 / (1 + np.linalg.norm(v1 - v2))
            man_sim = 100 / (1 + np.sum(np.abs(v1 - v2)))
            sim_matrix.loc[c1, c2] = round((cos + euc_sim + man_sim) / 3, 1)

sim_matrix.to_csv(f"{results_dir}/Similarity_Matrix.csv")

pairs = [{'Class1': c1, 'Class2': c2, 'Similarity': sim_matrix.loc[c1, c2]}
         for i, c1 in enumerate(label_names) for c2 in label_names[i+1:]]
pairs_df = pd.DataFrame(pairs).sort_values('Similarity', ascending=False)
pairs_df.head(15).to_csv(f"{results_dir}/Top15_Most_Similar_Pairs.csv", index=False)
pairs_df.tail(15).to_csv(f"{results_dir}/Top15_Most_Dissimilar_Pairs.csv", index=False)

n_classes = len(label_names)
cluster_results = []
for name, model in [
    ("KMeans", KMeans(n_clusters=n_classes, random_state=42, n_init=10)),
    ("Agglomerative_Ward", AgglomerativeClustering(n_clusters=n_classes, linkage='ward')),
    ("GaussianMixture", GaussianMixture(n_components=n_classes, random_state=42))
]:
    pred = model.fit_predict(X_scaled)
    cluster_results.append({
        'Method': name,
        'ARI': round(adjusted_rand_score(y_encoded, pred), 4),
        'NMI': round(normalized_mutual_info_score(y_encoded, pred), 4)
    })
cluster_df = pd.DataFrame(cluster_results)
cluster_df.to_csv(f"{results_dir}/Clustering_Performance.csv", index=False)   # <- Table S1
print("\nClustering Performance:")
print(cluster_df.to_string(index=False))

kmeans_pred = KMeans(n_clusters=n_classes, random_state=42, n_init=10).fit_predict(X_scaled)

# ==================== PAPER FIGURES ====================
print("\n[5/7] Generating figures...")

# 1. Similarity Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(sim_matrix, annot=True, fmt=".1f", cmap="RdYlBu_r", center=50,
            linewidths=0.5, cbar_kws={'label': 'Average Similarity (%)'})
plt.title(f"Pairwise Similarity Matrix - {DATASET_TYPE.upper()}", fontsize=14)
plt.tight_layout()
plt.savefig(f"{paper_dir}/Fig1_Similarity_Heatmap.png", dpi=300, bbox_inches='tight')
plt.close()

# 2. t-SNE with Outliers
tsne = TSNE(n_components=2, random_state=42, perplexity=40, max_iter=1000)
X_tsne = tsne.fit_transform(X_scaled)

plt.figure(figsize=(12, 9))
palette = sns.color_palette("tab10", n_colors=len(label_names))
for i, cls in enumerate(label_names):
    mask = (y == cls) & (df['is_outlier'] == 0)
    plt.scatter(X_tsne[mask, 0], X_tsne[mask, 1], color=palette[i], label=cls, s=20, alpha=0.7)
    mask_out = (y == cls) & (df['is_outlier'] == 1)
    if mask_out.sum() > 0:
        plt.scatter(X_tsne[mask_out, 0], X_tsne[mask_out, 1], color='red', s=70, marker='X',
                    edgecolors='black', linewidths=0.6, label=f'{cls} (outlier)' if i == 0 else "")
plt.title(f"t-SNE - {DATASET_TYPE.upper()} (Red X = Outliers)", fontsize=14)
plt.legend(title="Class", bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig(f"{paper_dir}/Fig2_tSNE_with_Outliers.png", dpi=300, bbox_inches='tight')
plt.close()

# 3. PCA
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
plt.figure(figsize=(12, 9))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=y, palette="tab10", s=25, alpha=0.75, edgecolor='none')
plt.title(f"PCA - {DATASET_TYPE.upper()} (Variance: {pca.explained_variance_ratio_.sum()*100:.1f}%)", fontsize=14)
plt.tight_layout()
plt.savefig(f"{paper_dir}/Fig3_PCA.png", dpi=300, bbox_inches='tight')
plt.close()

# 4. Clustering Bar Plot
plt.figure(figsize=(8, 5))
melted = cluster_df.melt(id_vars='Method', var_name='Metric', value_name='Score')
sns.barplot(data=melted, x='Method', y='Score', hue='Metric', palette="Set2")
plt.title(f"Clustering Performance - {DATASET_TYPE.upper()}", fontsize=13)
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig(f"{paper_dir}/Fig4_Clustering_Performance.png", dpi=300, bbox_inches='tight')
plt.close()

# 5. Cluster vs True Heatmap
cm = confusion_matrix(y_encoded, kmeans_pred, normalize='true')
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=label_names, yticklabels=label_names)
plt.xlabel("Predicted Cluster")
plt.ylabel("True Class")
plt.title(f"Cluster Assignment vs True Class - {DATASET_TYPE.upper()}", fontsize=13)
plt.tight_layout()
plt.savefig(f"{paper_dir}/Fig5_Cluster_vs_True_Heatmap.png", dpi=300, bbox_inches='tight')
plt.close()

# 6. Dendrogram
plt.figure(figsize=(14, 7))
class_mean_matrix = np.array([class_features[cls] for cls in label_names])
linked = linkage(class_mean_matrix, method='ward')
dendrogram(linked, labels=label_names, orientation='top', leaf_rotation=45)
plt.title(f"Hierarchical Clustering Dendrogram - {DATASET_TYPE.upper()}", fontsize=13)
plt.tight_layout()
plt.savefig(f"{paper_dir}/Fig6_Dendrogram.png", dpi=300, bbox_inches='tight')
plt.close()

print("All figures saved successfully!")
print("\n" + "="*80)
print(f"{DATASET_TYPE.upper()} ANALYSIS COMPLETE!  Results: {results_dir}")
print("="*80)